# Séries Temporais — Relatório Final Consolidado

## Quatro modelos, cinco bases e aula de PLS Regression

**Arquitetura de entrega — versão com Base Ouro concluída**

<span class="badge gold">Base 5 disponível</span>
<span class="badge pending">Bases 1–4 pendentes</span>
<span class="badge pending">Ranking global pendente</span>

Este documento é o material visual principal da apresentação. Campos pendentes
são deliberados: nenhum resultado ausente foi estimado ou inventado.


## 1. Resumo executivo

A atividade compara **SARIMAX, Holt-Winters, Random Forest e PLS Regression** em
cinco séries temporais. Cada base deve ser julgada pelo próprio teste temporal;
MAEs de escalas diferentes não são promediados diretamente.

Até esta versão, apenas a **Base 5 — Ouro** está consolidada. Seu ranking oficial
V1 é: Holt-Winters 11,5331; SARIMAX 11,5988; PLS 11,6002; Random Forest 14,2826.
A persistência, 11,6377, é baseline diagnóstico e não integra o ranking.

<div class="callout">Conclusões globais, vitórias e posição média permanecem
pendentes até a incorporação auditada das Bases 1–4.</div>


## 2. Integrantes e responsabilidades

<div class="pending-box"><strong>PENDENTE — equipe global</strong><br>
Inserir nomes, bases assumidas, responsável pela consolidação, responsável pela
aula de PLS e responsável pela apresentação. Não preencher sem o registro oficial
do grupo.</div>

| Responsabilidade | Integrante | Estado |
|---|---|---|
| Base 1 | PENDENTE | não informado |
| Base 2 | PENDENTE | não informado |
| Base 3 | PENDENTE | não informado |
| Base 4 | PENDENTE | não informado |
| Base 5 — Ouro | Grupo 5 | estudo de caso disponível |
| Consolidação e apresentação | PENDENTE | não informado |


## 3. Metodologia comum

1. Ordenar cada série no tempo e fixar o horizonte de previsão.
2. Construir features apenas com informação disponível na origem.
3. Ajustar pré-processamento e modelo dentro de cada janela de treino.
4. Selecionar hiperparâmetros na validação walk-forward.
5. Congelar a especificação e avaliar o teste final uma única vez.
6. Comparar os quatro modelos pelo MAE dentro da mesma base.
7. Complementar o erro com resíduos, ACF, Ljung-Box e interpretabilidade.

<div class="flow"><span>treino</span><b>→</b><span>validação temporal</span><b>→</b><span>congelamento</span><b>→</b><span>teste final</span></div>

O teste final não escolhe features, componentes ou hiperparâmetros. Informações
posteriores à origem não podem entrar no vetor preditor.


## 4. Bases da atividade

| Base | Série/fonte | Relatório local | Estado |
|---|---|---|---|
| Base 1 | PENDENTE | `../grupo1/` | não consolidada |
| Base 2 | PENDENTE | `../grupo2/` | não consolidada |
| Base 3 | PENDENTE | `../grupo3/` | não consolidada |
| Base 4 | PENDENTE | `../grupo4/` | não consolidada |
| Base 5 — Ouro | preço do ouro; Bundesbank | `../grupo5/` | V1 oficial concluída |

Na Base Ouro, a série institucional é `BBEX3.D.XAU.USD.EA.AC.C05`; o protocolo
registra a URL do arquivo usado e as séries externas. As demais fontes serão
incluídas somente quando fornecidas pelos responsáveis.


## 5. Aula — PLS Regression: motivação

A **Partial Least Squares Regression (PLS)** é útil quando há muitas variáveis
explicativas correlacionadas e o objetivo é prever uma resposta contínua. Em vez
de ajustar a regressão diretamente nas colunas originais, o método cria
componentes latentes e regressa `y` nesses componentes.

<div class="flow"><span>features X</span><b>→</b><span>scaling</span><b>→</b><span>componentes T</span><b>→</b><span>previsão y</span></div>

A redução é **supervisionada**: a relação entre `X` e `y` participa da construção
dos componentes.


## 5.1 Multicolinearidade

Em séries temporais, preço atual, lags e médias móveis frequentemente carregam
informação parecida. Isso pode tornar coeficientes de uma regressão comum
instáveis, mesmo quando a previsão é útil.

| Sintoma | Consequência | Resposta da PLS |
|---|---|---|
| features redundantes | sinais sensíveis | combina informação em componentes |
| escalas diferentes | uma unidade domina | exige scaling coerente |
| muitos preditores correlacionados | interpretação difícil | usa espaço latente menor |

PLS não transforma associação em causalidade e não elimina a validação.


## 5.2 Componentes latentes

Cada componente é uma combinação linear das features padronizadas:

<div class="formula">t<sub>k</sub> = X w<sub>k</sub></div>

`w_k` contém pesos; `t_k` é o score do componente. Componentes sucessivos
resumem informação ainda não capturada, buscando estrutura útil para a resposta.

<div class="callout">O componente é uma variável construída, não uma feature
observada nem uma causa econômica.</div>


## 5.3 PLS × PCA

| Critério | PCA | PLS Regression |
|---|---|---|
| Usa `y` para formar componentes? | não | sim |
| Objetivo | explicar variação de `X` | prever `y` com estrutura de `X` |
| Redução | não supervisionada | supervisionada |
| Maior variação | pode ser pouco preditiva | busca relevância para a resposta |

PCA e PLS respondem a perguntas diferentes. A escolha depende do objetivo, não
apenas da quantidade de variância retida.


## 5.4 Scaling e `n_components`

<div class="formula">z = (x − μ<sub>treino</sub>) / σ<sub>treino</sub></div>

Média e desvio são estimados **somente no treino** de cada janela.
`n_components` controla a dimensão latente: poucos componentes podem subajustar;
muitos podem reintroduzir ruído. A escolha ocorre na validação temporal.

<div class="callout">Escolher 11 de 12 componentes não significa explicar
91,7% da variância. Essa divisão conta dimensões e não mede variância explicada
pela PLS.</div>


## 5.5 PLS em séries temporais

Cada linha representa uma origem de previsão. Lags, janelas móveis, calendário e
externas temporalmente válidas transformam a história em `X`.

<div class="flow"><span>história até t</span><b>→</b><span>features causais</span><b>→</b><span>PLS no passado</span><b>→</b><span>ŷ para t+h</span></div>

A PLS não modela automaticamente autocorrelação, sazonalidade ou estado
temporal; essas estruturas precisam estar nas features e ser avaliadas fora da
amostra.


## 5.6 Walk-forward e leakage

No walk-forward, a origem avança e o modelo é reestimado sem enxergar o futuro.
O tuning compara candidatos apenas na validação; o teste final não decide
componentes, features ou scaling.

<div class="flow"><span>passado</span><b>→</b><span>fit scaler + PLS</span><b>→</b><span>validação</span><b>→</b><span>congelar</span><b>→</b><span>teste</span></div>

Leakage inclui ajustar o scaler na série completa, usar janelas que contêm o alvo
ou escolher `n_components` pelo MAE de teste.


## 5.7 Interpretação

| Ferramenta | O que responde | Cuidado |
|---|---|---|
| coeficiente padronizado | direção no modelo | colinearidade redistribui sinais |
| VIP | contribuição nos componentes | limiar é heurístico |
| permutation importance | aumento do erro ao embaralhar | correlacionadas dividem importância |

As leituras são complementares e descritivas. Concordância aumenta confiança;
divergência pede cautela. Nenhuma delas estabelece causalidade.


## 5.8 Comparação com os outros modelos

| Modelo | Força central | Limitação típica |
|---|---|---|
| PLS | redução supervisionada com preditores correlacionados | linearidade e dependência das features |
| SARIMAX | dinâmica temporal e estrutura de erros | especificação das ordens |
| Holt-Winters | nível, tendência e sazonalidade univariada | não usa exógenas |
| Random Forest | não linearidades e interações | não extrapola naturalmente novos níveis |

Não existe vencedor universal. Alvo, horizonte, regime, features e protocolo
determinam o desempenho fora da amostra.


## 6. Aplicação do PLS nas cinco bases

| Base | Configuração PLS | MAE oficial | Estado |
|---|---|---:|---|
| Base 1 | PENDENTE | — | não consolidada |
| Base 2 | PENDENTE | — | não consolidada |
| Base 3 | PENDENTE | — | não consolidada |
| Base 4 | PENDENTE | — | não consolidada |
| Base 5 — Ouro | 12 features; `n_components=11`; horizonte 1 | 11,6002 | V1 oficial |

No Ouro, a validação e o teste contêm 296 origens cada. O MAE de validação
2,9701 e o MAE de teste 11,6002 pertencem a janelas temporais distintas, na
mesma unidade. A mudança de regime e volatilidade contextualiza a diferença.


## 7. Comparação dos quatro modelos nas cinco bases

| Base | SARIMAX | Holt-Winters | Random Forest | PLS |
|---|---:|---:|---:|---:|
| Base 1 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 2 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 3 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 4 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 5 — Ouro | 11,5988 | **11,5331** | 14,2826 | 11,6002 |

O Random Forest do Ouro teve dificuldade de extrapolação: no diagnóstico V2 A8,
94,9% dos alvos de teste ficaram acima do máximo de treino. A V2 é explicativa e
não substitui os resultados oficiais.


## 8. Resultados — MAE e ranking por base

| Base | 1º | 2º | 3º | 4º |
|---|---|---|---|---|
| Base 1 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 2 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 3 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 4 | PENDENTE | PENDENTE | PENDENTE | PENDENTE |
| Base 5 — Ouro | **Holt-Winters — 11,5331** | SARIMAX — 11,5988 | PLS — 11,6002 | Random Forest — 14,2826 |

Persistência no Ouro: **11,6377**, usada como baseline diagnóstico e mantida fora
do ranking dos quatro modelos.


## 9. Ranking geral dos modelos

| Modelo | Vitórias | Posição média | Ranking global |
|---|---:|---:|---:|
| SARIMAX | PENDENTE | PENDENTE | PENDENTE |
| Holt-Winters | PENDENTE | PENDENTE | PENDENTE |
| Random Forest | PENDENTE | PENDENTE | PENDENTE |
| PLS Regression | PENDENTE | PENDENTE | PENDENTE |

<div class="pending-box"><strong>NÃO CALCULADO.</strong> O ranking global exige
as cinco bases. A comparação futura usará vitórias e posição média; não usará a
média direta dos MAEs, pois as escalas e unidades podem ser diferentes.</div>


## 10. Resíduos, ACF e Ljung-Box

O diagnóstico residual deve acompanhar o MAE em cada base. Para o Ouro, a V1
inclui resíduos completos, ACF e Ljung-Box. O PLS teve **0 rejeições** nos lags
avaliados; isso indica ausência de evidência de autocorrelação residual nesses
lags, não prova independência perfeita.

| Base | Resíduos/ACF/Ljung-Box | Estado |
|---|---|---|
| Bases 1–4 | PENDENTE | não consolidado |
| Base 5 — Ouro | disponível no relatório local e apêndice | concluído |


## 11. Feature importance e interpretabilidade

No Ouro, preço atual, lag 1, média móvel e lags 5/20 concentram a importância.
A leitura é compatível com forte memória recente no horizonte 1, mas não é uma
afirmação causal. A redundância entre features pode repartir importância.

A V2 compara alvos `LEVEL`, `DELTA` e `LOG_RETURN`, ablation e drift. O menor MAE
experimental foi RF/LOG_RETURN/A3, 11,2986, classificado como `INFORMATIVE`.
Ele não altera o ranking oficial V1.


## 12. Conclusões disponíveis

Para a Base Ouro, Holt-Winters obteve o menor MAE oficial, seguido de SARIMAX e
PLS com diferenças pequenas; Random Forest ficou atrás. O PLS foi competitivo e
apresentou diagnóstico residual favorável, mas um único estudo de caso não
estabelece superioridade geral.

<div class="pending-box"><strong>Conclusão global pendente.</strong> Não é
possível afirmar qual modelo vence a atividade, quantas vitórias cada um tem ou
qual é sua posição média antes das Bases 1–4.</div>


## 13. Limitações e recomendações

- quatro bases ainda não foram incorporadas ao consolidado;
- MAE depende de escala, horizonte e regime da série;
- a leitura de importância não é causal;
- árvores não extrapolam naturalmente regimes de nível inéditos;
- o diagnóstico V2 do Ouro é complementar, com
  `PASS_WITH_METADATA_RECONCILIATION_REQUIRED` para `target_date`;
- a reconciliação corrige rótulo/metadado e não altera previsões ou MAEs;
- cada nova base deve entrar com protocolo, tabela de MAE, resíduos, referências
  e rastreabilidade equivalentes.


## 14. Referências e rastreabilidade

As referências consolidadas estão em `referencias.md`; as fontes completas do
Ouro, em `../grupo5/referencias.md`. O arquivo `resultados_mae.csv` mantém as 20
combinações base-modelo, com placeholders explícitos para resultados ausentes.

O apêndice técnico está em `apendice_tecnico.md` e o registro diário em
`registro_demandas.md`. A fonte deste material é `relatorio_final.ipynb`.

<div class="callout">Versão atual: arquitetura pronta para integração; conteúdo
global ainda incompleto por dependência das Bases 1–4 e da equipe.</div>
